# DCOPF Example - Per Unit (Data from File)
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (M3 vs M2):** All MW quantities (load, generation limits, branch ratings) are normalised to per-unit (pu) by dividing by `BaseMW` before building the model. The line flow equation is therefore `pk = (theta_from - theta_to) / x` (no `/BaseMW` term). The objective multiplies back by `BaseMW` so the cost remains in $/h.

In [ ]:
from pyomo.environ import *
import re
import time

# ─────────────────────────────────────────────────────────────────
# Base path & data file
# ─────────────────────────────────────────────────────────────────
BASE_PATH = '/Users/csab/Desktop/ECE6379_Python_Code/20_DCOPF/'
data_file = BASE_PATH + 'DCOPF_IC_e1_M2M3_data.txt'

# Fixed system parameters
BaseMW = 100
refBus = 3

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Parser for AMPL-style data files
# Supports:
#   1D param block : param: SET_NAME: col1 col2 ... :=
#   2D param block : param NAME: col1 col2 ... :=  (matrix form)
#   Inline comments: # ...
# ─────────────────────────────────────────────────────────────────
def parse_ampl_data(filepath):
    with open(filepath, 'r') as f:
        raw_lines = f.readlines()

    clean_lines = [re.sub(r'#.*', '', l).strip() for l in raw_lines]
    clean_lines = [l for l in clean_lines if l]
    text = ' '.join(clean_lines)

    blocks = [b.strip() for b in re.split(r';', text) if b.strip()]

    sets   = {}
    params = {}

    for block in blocks:
        # ── 1D block: param: SET_NAME: col1 col2 ... := row_data ──
        m = re.match(r'param\s*:\s*(\w+)\s*:\s*([\w\s]+):=\s*(.*)', block, re.S)
        if m:
            set_name  = m.group(1).strip()
            col_names = m.group(2).split()
            tokens    = m.group(3).split()
            idx_list  = []
            for col in col_names:
                params.setdefault(col, {})
            it = iter(tokens)
            try:
                while True:
                    idx = int(next(it))
                    idx_list.append(idx)
                    for col in col_names:
                        params[col][idx] = float(next(it))
            except StopIteration:
                pass
            sets[set_name] = idx_list
            continue

        # ── 2D block: param NAME: col1 col2 ... := row_data ───────
        m = re.match(r'param\s+(\w+)\s*:\s*([\d\s]+):=\s*(.*)', block, re.S)
        if m:
            param_name  = m.group(1).strip()
            col_indices = [int(c) for c in m.group(2).split()]
            tokens      = m.group(3).split()
            param_dict  = {}
            it = iter(tokens)
            try:
                while True:
                    row_idx = int(next(it))
                    for col_idx in col_indices:
                        param_dict[(row_idx, col_idx)] = float(next(it))
            except StopIteration:
                pass
            params[param_name] = param_dict
            continue

    return sets, params


# ─────────────────────────────────────────────────────────────────
# Load & normalise data (per-unit conversion)
# ─────────────────────────────────────────────────────────────────
sets, params = parse_ampl_data(data_file)

BUS_data    = sets['BUS']
GEN_data    = sets['GEN']
BRANCH_data = sets['BRANCH']

# Raw MW values
busLoad_data       = {int(k): v       for k, v in params['busLoad'].items()}
gen_bus_data       = {int(k): int(v)  for k, v in params['gen_bus'].items()}
gen_min_data       = {int(k): v       for k, v in params['gen_min'].items()}
gen_max_data       = {int(k): v       for k, v in params['gen_max'].items()}
gen_OpCost_data    = {int(k): v       for k, v in params['gen_OpCost'].items()}
branch_frmbus_data = {int(k): int(v)  for k, v in params['branch_frmbus'].items()}
branch_tobus_data  = {int(k): int(v)  for k, v in params['branch_tobus'].items()}
branch_x_data      = {int(k): v       for k, v in params['branch_x'].items()}
branch_rate_data   = {int(k): v       for k, v in params['branch_rate'].items()}

# ── Per-unit normalisation (divide MW quantities by BaseMW) ──────
busLoad_pu    = {k: v / BaseMW for k, v in busLoad_data.items()}
gen_min_pu    = {k: v / BaseMW for k, v in gen_min_data.items()}
gen_max_pu    = {k: v / BaseMW for k, v in gen_max_data.items()}
branch_rate_pu = {k: v / BaseMW for k, v in branch_rate_data.items()}
# gen_OpCost stays in $/MWh; objective multiplied by BaseMW to recover $/h

print('BUS    :', BUS_data)
print('GEN    :', GEN_data)
print('BRANCH :', BRANCH_data)
print('busLoad (pu):', busLoad_pu)
print('gen limits (pu):', {g: (gen_min_pu[g], gen_max_pu[g]) for g in GEN_data})
print('branch_rate (pu):', branch_rate_pu)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model (per-unit formulation)
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.BUS    = Set(initialize=BUS_data)
model.BRANCH = Set(initialize=BRANCH_data)
model.GEN    = Set(initialize=GEN_data)

# Parameters (pu where applicable)
model.busLoad       = Param(model.BUS,    initialize=busLoad_pu)
model.gen_bus       = Param(model.GEN,    initialize=gen_bus_data,        within=model.BUS)
model.gen_min       = Param(model.GEN,    initialize=gen_min_pu)
model.gen_max       = Param(model.GEN,    initialize=gen_max_pu)
model.gen_OpCost    = Param(model.GEN,    initialize=gen_OpCost_data)      # $/MWh
model.branch_frmbus = Param(model.BRANCH, initialize=branch_frmbus_data,   within=model.BUS)
model.branch_tobus  = Param(model.BRANCH, initialize=branch_tobus_data,    within=model.BUS)
model.branch_x      = Param(model.BRANCH, initialize=branch_x_data)        # pu
model.branch_rate   = Param(model.BRANCH, initialize=branch_rate_pu)       # pu

# Decision Variables (all in pu)
model.G     = Var(model.GEN)
model.pk    = Var(model.BRANCH)
model.theta = Var(model.BUS)

# ── Objective: minimise total cost ($/h) ─────────────────────────
# G is in pu, multiply by BaseMW to get MW, then times cost rate
def obj_rule(model):
    return sum(model.gen_OpCost[g] * model.G[g] * BaseMW for g in model.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Constraints ──────────────────────────────────────────────────
def branchLimits_rule(model, k):
    return inequality(-model.branch_rate[k], model.pk[k], model.branch_rate[k])
model.branchLimits = Constraint(model.BRANCH, rule=branchLimits_rule)

# Per-unit line flow: pk = (theta_from - theta_to) / x  (no BaseMW)
def lineFlowEqs_rule(model, k):
    return model.pk[k] == (
        model.theta[model.branch_frmbus[k]] - model.theta[model.branch_tobus[k]]
    ) / model.branch_x[k]
model.lineFlowEqs = Constraint(model.BRANCH, rule=lineFlowEqs_rule)

def genLimits_rule(model, g):
    return inequality(model.gen_min[g], model.G[g], model.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

def NodalPowerBalance_rule(model, n):
    gen_power = sum(model.G[g]  for g in model.GEN    if model.gen_bus[g]       == n)
    power_in  = sum(model.pk[k] for k in model.BRANCH if model.branch_tobus[k]  == n)
    power_out = sum(model.pk[k] for k in model.BRANCH if model.branch_frmbus[k] == n)
    return gen_power + power_in - power_out == model.busLoad[n]
model.NodalPowerBalance = Constraint(model.BUS, rule=NodalPowerBalance_rule)

# Fix reference bus angle
model.theta[refBus].fix(0)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\nG (pu):')
for g in model.GEN:
    print(f'  G[{g}] = {model.G[g].value:.6f} pu  ({model.G[g].value * BaseMW:.4f} MW)')

print('\npk (pu):')
for k in model.BRANCH:
    print(f'  pk[{k}] = {model.pk[k].value:.6f} pu  ({model.pk[k].value * BaseMW:.4f} MW)')

print('\ntheta (rad):')
for n in model.BUS:
    print(f'  theta[{n}] = {model.theta[n].value:.6f} rad')

print(f'\nTotal solve elapsed time: {solve_time:.4f} seconds')